# PhyloP Conservation Analysis - Complete Workflow

**Complete end-to-end analysis of translon conservation using PhyloP scores**

## Key Features:
- Handles spliced features properly (multi-exon translons)
- Uses **exonic flanks** (walks along transcript exons, skips introns)
- Calculates CDS overlap by proportion
- Identifies translons with restricted conservation

## Outputs:
1. Basic story file (65 restricted candidates + others)
2. Basic column descriptions
3. Comprehensive results (all statistics)
4. Comprehensive column descriptions

## Setup and Installation

In [ ]:
# Install required packages
import sys

try:
    import pyBigWig
    import pyranges as pr
    print("All packages already installed")
except ImportError:
    print("Installing required packages...")
    !{sys.executable} -m pip install --user pyBigWig pyranges
    print("\nInstallation complete. Please restart kernel.")
    print("After restart, run this cell again to verify installation.")

In [ ]:
import pandas as pd
import numpy as np
import pyBigWig
import pyranges as pr
import gzip
from collections import defaultdict
from pathlib import Path

print("All imports successful")

## Configuration

In [ ]:
# Input files
BIGBED_URL = 'https://ftp.ebi.ac.uk/pub/databases/gencode/riboseq_orfs/data/Ribo-seq_ORFs.bb'
BIGBED_FILE = '../data/Ribo-seq_ORFs.bb'
TRANSCRIPT_ANNOTATIONS = '../../phase1_w_transcript.tsv'
GENCODE_GTF = '../../data/gencode.v46.annotation.gtf.gz'
PHYLOP_470WAY = '../../data/phylop/hg38.phyloP470way.bw'

# Output directory
OUTPUT_DIR = Path('../results/phylop')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Parameters
FLANK_SIZE = 150  # bases of EXONIC sequence for flanks
CONSERVATION_THRESHOLD = 1.5  # PhyloP threshold for "strong conservation"
FLANK_THRESHOLD = 1.0  # Max flank threshold for "restricted conservation"

print(f"Configuration:")
print(f"  Flank size: {FLANK_SIZE}bp (exonic)")
print(f"  Conservation threshold: {CONSERVATION_THRESHOLD}")
print(f"  Flank threshold: {FLANK_THRESHOLD}")
print(f"  Output directory: {OUTPUT_DIR}")

## Step 1: Load Transcript Exon Structures from GENCODE

In [ ]:
print("Loading transcript exon structures from GENCODE GTF...")

transcript_exons = defaultdict(list)

with gzip.open(GENCODE_GTF, 'rt') as f:
    for line in f:
        if line.startswith('#'):
            continue
        
        fields = line.strip().split('\t')
        if len(fields) < 9 or fields[2] != 'exon':
            continue
        
        chrom = fields[0]
        start = int(fields[3]) - 1  # GTF is 1-based, convert to 0-based
        end = int(fields[4])
        strand = fields[6]
        
        # Parse attributes for transcript_id
        attrs = {}
        for attr in fields[8].split(';'):
            attr = attr.strip()
            if attr:
                parts = attr.split(' ', 1)
                if len(parts) == 2:
                    key, val = parts
                    attrs[key] = val.strip('"')
        
        transcript_id = attrs.get('transcript_id', '')
        if not transcript_id:
            continue
        
        transcript_exons[transcript_id].append({
            'chrom': chrom,
            'start': start,
            'end': end,
            'strand': strand
        })

# Sort exons by start position
for transcript_id in transcript_exons:
    transcript_exons[transcript_id].sort(key=lambda x: x['start'])

print(f"Loaded exons for {len(transcript_exons):,} transcripts")

## Step 2: Load Translon Data and Merge with Transcript Annotations

In [ ]:
# Check if BigBed file exists, download if needed
import os

if not os.path.exists(BIGBED_FILE):
    print(f"Downloading BigBed file from {BIGBED_URL}...")
    !curl -o {BIGBED_FILE} {BIGBED_URL}
    print(f"Downloaded to {BIGBED_FILE}")
else:
    print(f"BigBed file already exists: {BIGBED_FILE}")

In [ ]:
# Convert BigBed to BED
bed_file = BIGBED_FILE.replace('.bb', '.bed')

if not os.path.exists(bed_file):
    print(f"Converting BigBed to BED...")
    !bigBedToBed {BIGBED_FILE} {bed_file}
    print(f"Converted to {bed_file}")
else:
    print(f"BED file already exists: {bed_file}")

# Load BED file
print(f"\nLoading translons from {bed_file}...")
bed_df = pd.read_csv(
    bed_file,
    sep='\t',
    header=None,
    names=['chrom', 'start', 'end', 'name', 'score', 'strand',
           'thickStart', 'thickEnd', 'itemRgb', 'blockCount',
           'blockSizes', 'blockStarts'],
    dtype={'blockSizes': str, 'blockStarts': str}
)

bed_df['translon_id'] = bed_df['name']
print(f"Loaded {len(bed_df):,} translons from BigBed")

# Load transcript annotations
print(f"\nLoading transcript annotations from {TRANSCRIPT_ANNOTATIONS}...")
transcript_df = pd.read_csv(TRANSCRIPT_ANNOTATIONS, sep='\t')
print(f"Loaded {len(transcript_df):,} translon-transcript mappings")

# Merge
translons = bed_df.merge(
    transcript_df[['orf_name', 'transcript']],
    left_on='translon_id',
    right_on='orf_name',
    how='left'
)

# Calculate exonic length
def calc_exonic_length(row):
    if pd.isna(row['blockSizes']):
        return row['end'] - row['start']
    sizes = [int(x) for x in str(row['blockSizes']).rstrip(',').split(',')]
    return sum(sizes)

translons['exonic_length'] = translons.apply(calc_exonic_length, axis=1)

print(f"\nMerged dataset:")
print(f"  Total translons: {len(translons):,}")
print(f"  With transcript ID: {translons['transcript'].notna().sum():,}")
print(f"  Without transcript ID: {translons['transcript'].isna().sum():,}")
print(f"  Single-exon: {(translons['blockCount'] == 1).sum():,}")
print(f"  Multi-exon: {(translons['blockCount'] > 1).sum():,}")

## Step 3: Define Helper Functions for Exonic Flank Extraction

In [ ]:
def parse_bed_blocks(chrom_start, block_count, block_sizes_str, block_starts_str):
    """
    Parse BED12 block notation into list of (start, end) tuples.
    """
    if pd.isna(block_sizes_str) or pd.isna(block_starts_str):
        return []
    
    try:
        block_sizes = [int(x) for x in str(block_sizes_str).rstrip(',').split(',')]
        block_starts = [int(x) for x in str(block_starts_str).rstrip(',').split(',')]
        
        blocks = []
        for i in range(int(block_count)):
            block_start = chrom_start + block_starts[i]
            block_end = block_start + block_sizes[i]
            blocks.append((block_start, block_end))
        
        return blocks
    except:
        return []


def get_exonic_flanks(translon_chrom, translon_blocks, translon_strand, transcript_id, flank_size=150):
    """
    Get exonic flanking regions by walking along transcript exons.
    Skips introns completely.
    
    Returns:
        upstream_coords: list of (start, end) tuples
        downstream_coords: list of (start, end) tuples
    """
    if not translon_blocks:
        return [], []
    
    translon_min = min(s for s, e in translon_blocks)
    translon_max = max(e for s, e in translon_blocks)
    
    # Get transcript exons
    exons = transcript_exons.get(transcript_id, [])
    
    # Fallback to genomic flanks if no transcript annotation
    if not exons or pd.isna(transcript_id):
        if translon_strand == '+':
            upstream_coords = [(max(0, translon_min - flank_size), translon_min)]
            downstream_coords = [(translon_max, translon_max + flank_size)]
        else:
            upstream_coords = [(translon_max, translon_max + flank_size)]
            downstream_coords = [(max(0, translon_min - flank_size), translon_min)]
        return upstream_coords, downstream_coords
    
    # Separate exons: before, overlapping, after translon
    before_exons = [e for e in exons if e['end'] <= translon_min]
    after_exons = [e for e in exons if e['start'] >= translon_max]
    translon_exons = [e for e in exons if not (e['end'] <= translon_min or e['start'] >= translon_max)]
    
    strand = exons[0]['strand']
    
    if strand == '+':
        # Upstream: walk backwards collecting exonic bases
        upstream_coords = []
        remaining = flank_size
        
        # Check translon exons for bases before translon
        for exon in reversed(translon_exons):
            if remaining <= 0:
                break
            if exon['start'] < translon_min:
                take_start = max(exon['start'], translon_min - remaining)
                take_end = translon_min
                upstream_coords.insert(0, (take_start, take_end))
                remaining -= (take_end - take_start)
        
        # Walk backwards through before_exons
        for exon in reversed(before_exons):
            if remaining <= 0:
                break
            exon_len = exon['end'] - exon['start']
            if exon_len <= remaining:
                upstream_coords.insert(0, (exon['start'], exon['end']))
                remaining -= exon_len
            else:
                upstream_coords.insert(0, (exon['end'] - remaining, exon['end']))
                remaining = 0
        
        # Extend genomically if needed
        if remaining > 0:
            transcript_start = min(e['start'] for e in exons)
            upstream_coords.insert(0, (max(0, transcript_start - remaining), transcript_start))
        
        # Downstream: walk forwards
        downstream_coords = []
        remaining = flank_size
        
        # Check translon exons for bases after translon
        for exon in translon_exons:
            if remaining <= 0:
                break
            if exon['end'] > translon_max:
                take_start = translon_max
                take_end = min(exon['end'], translon_max + remaining)
                downstream_coords.append((take_start, take_end))
                remaining -= (take_end - take_start)
        
        # Walk forwards through after_exons
        for exon in after_exons:
            if remaining <= 0:
                break
            exon_len = exon['end'] - exon['start']
            if exon_len <= remaining:
                downstream_coords.append((exon['start'], exon['end']))
                remaining -= exon_len
            else:
                downstream_coords.append((exon['start'], exon['start'] + remaining))
                remaining = 0
        
        # Extend genomically if needed
        if remaining > 0:
            transcript_end = max(e['end'] for e in exons)
            downstream_coords.append((transcript_end, transcript_end + remaining))
    
    else:  # strand == '-'
        # For minus strand: upstream (5' of gene) is genomically downstream
        upstream_coords = []
        remaining = flank_size
        
        # Check translon exons for bases after translon (genomically)
        for exon in translon_exons:
            if remaining <= 0:
                break
            if exon['end'] > translon_max:
                take_start = translon_max
                take_end = min(exon['end'], translon_max + remaining)
                upstream_coords.append((take_start, take_end))
                remaining -= (take_end - take_start)
        
        # Walk through after_exons
        for exon in after_exons:
            if remaining <= 0:
                break
            exon_len = exon['end'] - exon['start']
            if exon_len <= remaining:
                upstream_coords.append((exon['start'], exon['end']))
                remaining -= exon_len
            else:
                upstream_coords.append((exon['start'], exon['start'] + remaining))
                remaining = 0
        
        # Extend genomically if needed
        if remaining > 0:
            transcript_end = max(e['end'] for e in exons)
            upstream_coords.append((transcript_end, transcript_end + remaining))
        
        # Downstream (3' of gene) = genomically upstream
        downstream_coords = []
        remaining = flank_size
        
        # Check translon exons for bases before translon
        for exon in reversed(translon_exons):
            if remaining <= 0:
                break
            if exon['start'] < translon_min:
                take_start = max(exon['start'], translon_min - remaining)
                take_end = translon_min
                downstream_coords.insert(0, (take_start, take_end))
                remaining -= (take_end - take_start)
        
        # Walk backwards through before_exons
        for exon in reversed(before_exons):
            if remaining <= 0:
                break
            exon_len = exon['end'] - exon['start']
            if exon_len <= remaining:
                downstream_coords.insert(0, (exon['start'], exon['end']))
                remaining -= exon_len
            else:
                downstream_coords.insert(0, (exon['end'] - remaining, exon['end']))
                remaining = 0
        
        # Extend genomically if needed
        if remaining > 0:
            transcript_start = min(e['start'] for e in exons)
            downstream_coords.insert(0, (max(0, transcript_start - remaining), transcript_start))
    
    return upstream_coords, downstream_coords


def extract_phylop_scores(bw, chrom, coord_list):
    """Extract PhyloP scores from multiple coordinate ranges."""
    all_scores = []
    
    for start, end in coord_list:
        scores = bw.values(chrom, start, end, numpy=True)
        if scores is not None:
            valid_scores = scores[~np.isnan(scores)]
            if len(valid_scores) > 0:
                all_scores.extend(valid_scores)
    
    return np.array(all_scores) if all_scores else np.array([])


def calc_stats(scores):
    """Calculate statistics for PhyloP scores."""
    if len(scores) == 0:
        return {
            'n_bases': 0,
            'mean': np.nan,
            'median': np.nan,
            'std': np.nan,
            'min': np.nan,
            'max': np.nan
        }
    return {
        'n_bases': len(scores),
        'mean': np.mean(scores),
        'median': np.median(scores),
        'std': np.std(scores),
        'min': np.min(scores),
        'max': np.max(scores)
    }

print("Helper functions defined")

## Step 4: Extract PhyloP Scores with Exonic Flanks

In [ ]:
print(f"Opening PhyloP 470way: {PHYLOP_470WAY}")
bw = pyBigWig.open(PHYLOP_470WAY)

results = []

for idx, row in translons.iterrows():
    if idx % 1000 == 0:
        print(f"Processing {idx+1:,}/{len(translons):,}...")
    
    translon_id = row['translon_id']
    chrom = row['chrom']
    start = row['start']
    end = row['end']
    strand = row['strand']
    transcript_id = row.get('transcript', None)
    
    # Parse translon blocks
    block_count = row.get('blockCount', 1)
    block_sizes = row.get('blockSizes', '')
    block_starts = row.get('blockStarts', '')
    
    translon_blocks = parse_bed_blocks(start, block_count, block_sizes, block_starts)
    if not translon_blocks:
        translon_blocks = [(start, end)]
    
    # Get exonic flanks
    upstream_coords, downstream_coords = get_exonic_flanks(
        chrom, translon_blocks, strand, transcript_id, FLANK_SIZE
    )
    
    # Extract PhyloP scores
    feature_scores = extract_phylop_scores(bw, chrom, translon_blocks)
    upstream_scores = extract_phylop_scores(bw, chrom, upstream_coords)
    downstream_scores = extract_phylop_scores(bw, chrom, downstream_coords)
    
    # Calculate statistics
    feature_stats = calc_stats(feature_scores)
    upstream_stats = calc_stats(upstream_scores)
    downstream_stats = calc_stats(downstream_scores)
    
    # Derived metrics
    feature_mean = feature_stats['mean']
    upstream_mean = upstream_stats['mean']
    downstream_mean = downstream_stats['mean']
    
    if not np.isnan(feature_mean) and not np.isnan(upstream_mean) and not np.isnan(downstream_mean):
        conservation_specificity = feature_mean - max(upstream_mean, downstream_mean)
    else:
        conservation_specificity = np.nan
    
    # Store result
    result = {
        'translon_id': translon_id,
        'chrom': chrom,
        'start': start,
        'end': end,
        'strand': strand,
        'transcript_id': transcript_id if pd.notna(transcript_id) else 'no_annotation',
        'exonic_length': row['exonic_length'],
        'blockCount': block_count,
        'flank_size': FLANK_SIZE,
        'flank_type': 'exonic',
        
        'feature_n_bases': feature_stats['n_bases'],
        'feature_mean': feature_stats['mean'],
        'feature_median': feature_stats['median'],
        'feature_std': feature_stats['std'],
        'feature_min': feature_stats['min'],
        'feature_max': feature_stats['max'],
        
        'upstream_n_bases': upstream_stats['n_bases'],
        'upstream_mean': upstream_stats['mean'],
        'upstream_median': upstream_stats['median'],
        'upstream_std': upstream_stats['std'],
        'upstream_min': upstream_stats['min'],
        'upstream_max': upstream_stats['max'],
        
        'downstream_n_bases': downstream_stats['n_bases'],
        'downstream_mean': downstream_stats['mean'],
        'downstream_median': downstream_stats['median'],
        'downstream_std': downstream_stats['std'],
        'downstream_min': downstream_stats['min'],
        'downstream_max': downstream_stats['max'],
        
        'conservation_specificity': conservation_specificity,
    }
    
    results.append(result)

bw.close()

df_results = pd.DataFrame(results)

print(f"\nCompleted processing {len(df_results):,} translons")
print(f"\nMean PhyloP scores:")
print(f"  Feature:    {df_results['feature_mean'].mean():6.3f}")
print(f"  Upstream:   {df_results['upstream_mean'].mean():6.3f}")
print(f"  Downstream: {df_results['downstream_mean'].mean():6.3f}")

## Step 5: Add CDS Overlap Analysis

In [ ]:
print("Extracting CDS features from GENCODE GTF...")

cds_records = []

with gzip.open(GENCODE_GTF, 'rt') as f:
    for line in f:
        if line.startswith('#'):
            continue
        
        fields = line.strip().split('\t')
        if len(fields) < 9 or fields[2] != 'CDS':
            continue
        
        chrom = fields[0]
        start = int(fields[3]) - 1
        end = int(fields[4])
        strand = fields[6]
        
        attrs = {}
        for attr in fields[8].split(';'):
            attr = attr.strip()
            if attr:
                parts = attr.split(' ', 1)
                if len(parts) == 2:
                    key, val = parts
                    attrs[key] = val.strip('"')
        
        cds_records.append({
            'Chromosome': chrom,
            'Start': start,
            'End': end,
            'Strand': strand,
            'gene_name': attrs.get('gene_name', ''),
        })

print(f"Found {len(cds_records):,} CDS features")

# Create PyRanges
cds_pr = pr.PyRanges(pd.DataFrame(cds_records))

translon_records = []
for _, row in df_results.iterrows():
    translon_records.append({
        'Chromosome': row['chrom'],
        'Start': row['start'],
        'End': row['end'],
        'Strand': row['strand'],
        'translon_id': row['translon_id']
    })

translon_pr = pr.PyRanges(pd.DataFrame(translon_records))

print("Finding overlaps...")
overlaps = translon_pr.join(cds_pr, how='left', suffix='_cds')

# Process overlaps and calculate proportion
overlap_results = []
for idx, row in df_results.iterrows():
    translon_id = row['translon_id']
    translon_len = row['end'] - row['start']
    
    translon_overlaps = overlaps.df[
        (overlaps.df['Chromosome'] == row['chrom']) &
        (overlaps.df['Start'] == row['start']) &
        (overlaps.df['End'] == row['end'])
    ]
    
    if len(translon_overlaps) > 0 and 'Start_cds' in translon_overlaps.columns:
        has_overlap = translon_overlaps['Start_cds'].notna().any()
        
        if has_overlap:
            overlapping_genes = translon_overlaps['gene_name'].dropna().unique()
            same_strand = (translon_overlaps['Strand'] == translon_overlaps['Strand_cds']).any()
            
            # Calculate overlap proportion
            overlap_bases = 0
            for _, overlap_row in translon_overlaps.iterrows():
                if pd.notna(overlap_row['Start_cds']):
                    overlap_start = max(row['start'], overlap_row['Start_cds'])
                    overlap_end = min(row['end'], overlap_row['End_cds'])
                    if overlap_end > overlap_start:
                        overlap_bases += (overlap_end - overlap_start)
            
            overlap_proportion = min(1.0, overlap_bases / translon_len) if translon_len > 0 else 0.0
            
            overlap_results.append({
                'translon_id': translon_id,
                'has_cds_overlap': True,
                'same_strand': same_strand,
                'cds_overlap_proportion': overlap_proportion,
                'overlapping_genes': ','.join(overlapping_genes[:5]) if len(overlapping_genes) > 0 else '-1'
            })
        else:
            overlap_results.append({
                'translon_id': translon_id,
                'has_cds_overlap': False,
                'same_strand': False,
                'cds_overlap_proportion': 0.0,
                'overlapping_genes': '-1'
            })
    else:
        overlap_results.append({
            'translon_id': translon_id,
            'has_cds_overlap': False,
            'same_strand': False,
            'cds_overlap_proportion': 0.0,
            'overlapping_genes': '-1'
        })

overlap_df = pd.DataFrame(overlap_results)

# Correct for PyRanges -1 sentinel
overlap_df['has_cds_overlap'] = overlap_df['overlapping_genes'] != '-1'

# Merge with results
df_results = df_results.merge(overlap_df, on='translon_id', how='left')

print(f"\nCDS overlap results:")
print(f"  With CDS overlap: {df_results['has_cds_overlap'].sum():,} ({100*df_results['has_cds_overlap'].sum()/len(df_results):.1f}%)")
print(f"  NO CDS overlap:   {(~df_results['has_cds_overlap']).sum():,} ({100*(~df_results['has_cds_overlap']).sum()/len(df_results):.1f}%)")
print(f"\nMean CDS overlap proportion: {df_results[df_results['has_cds_overlap']]['cds_overlap_proportion'].mean():.2%}")

## Step 6: Identify Restricted Conservation Candidates

In [ ]:
print("="*80)
print("IDENTIFYING RESTRICTED CONSERVATION CANDIDATES")
print("="*80)

# Filter for conserved translons
conserved = df_results[df_results['feature_mean'] > CONSERVATION_THRESHOLD]
conserved_no_cds = conserved[~conserved['has_cds_overlap']]

print(f"\nTotal translons: {len(df_results):,}")
print(f"Conserved (PhyloP >{CONSERVATION_THRESHOLD}): {len(conserved):,} ({100*len(conserved)/len(df_results):.1f}%)")
print(f"  With CDS overlap: {conserved['has_cds_overlap'].sum():,}")
print(f"  Without CDS overlap: {len(conserved_no_cds):,}")

# Calculate max flank for restricted conservation analysis
conserved_no_cds = conserved_no_cds.copy()
conserved_no_cds['max_flank'] = conserved_no_cds[['upstream_mean', 'downstream_mean']].max(axis=1)

print(f"\nMean max_flank for conserved without CDS: {conserved_no_cds['max_flank'].mean():.3f}")

# Apply threshold
restricted = conserved_no_cds[conserved_no_cds['max_flank'] <= FLANK_THRESHOLD]
regional = conserved_no_cds[conserved_no_cds['max_flank'] > FLANK_THRESHOLD]

print(f"\nREGIONAL conservation (exonic flanks >{FLANK_THRESHOLD}):")
print(f"  Count: {len(regional):,} ({100*len(regional)/len(conserved_no_cds):.1f}%)")
print(f"  Mean feature PhyloP: {regional['feature_mean'].mean():.3f}")
print(f"  Mean max_flank: {regional['max_flank'].mean():.3f}")

print(f"\nRESTRICTED conservation (exonic flanks ≤{FLANK_THRESHOLD}):")
print(f"  Count: {len(restricted):,} ({100*len(restricted)/len(conserved_no_cds):.1f}%)")
print(f"  Mean feature PhyloP: {restricted['feature_mean'].mean():.3f}")
print(f"  Mean max_flank: {restricted['max_flank'].mean():.3f}")

print(f"\n" + "="*80)
print("KEY FINDING FOR MANUSCRIPT")
print("="*80)
print(f"\nOf {len(df_results):,} translons, {len(restricted):,} ({100*len(restricted)/len(df_results):.1f}%) show:")
print(f"  1. Strong conservation (PhyloP >{CONSERVATION_THRESHOLD})")
print(f"  2. Independent of known CDS")
print(f"  3. Specific to feature (exonic flanks ≤{FLANK_THRESHOLD})")

## Step 7: Save Intermediate File (Comprehensive Results)

In [ ]:
# Save comprehensive results
comprehensive_output = OUTPUT_DIR / 'translon_phylop_v3_with_cds_overlap.tsv'
df_results.to_csv(comprehensive_output, sep='\t', index=False)

print(f"Saved comprehensive results: {comprehensive_output}")
print(f"  Rows: {len(df_results):,}")
print(f"  Columns: {len(df_results.columns)}")

## Step 8: Generate Final Output Files

In [ ]:
print("Generating final output files...\n")

# ============================================================================
# File 1: Basic story file (key columns only)
# ============================================================================

key_columns = [
    'translon_id', 'chrom', 'start', 'end', 'exonic_length', 'blockCount',
    'feature_mean', 'upstream_mean', 'downstream_mean',
    'conservation_specificity', 'has_cds_overlap', 'cds_overlap_proportion',
    'overlapping_genes'
]

# Sort restricted by specificity
restricted_subset = restricted[key_columns].copy()
restricted_subset['category'] = 'restricted_conservation'
restricted_sorted = restricted_subset.sort_values('conservation_specificity', ascending=False)

# Get other translons
other_translons = df_results[~df_results['translon_id'].isin(restricted['translon_id'])][key_columns].copy()
other_translons['category'] = 'other'
other_sorted = other_translons.sort_values('translon_id')

# Combine
final_basic = pd.concat([restricted_sorted, other_sorted], ignore_index=True)

output_basic = OUTPUT_DIR / 'v3_translon_conservation_story.tsv'
final_basic.to_csv(output_basic, sep='\t', index=False)

print(f"1. BASIC STORY FILE: {output_basic}")
print(f"   - {len(restricted_sorted):,} restricted conservation candidates on top")
print(f"   - {len(other_sorted):,} other translons below")
print(f"   - {len(key_columns) + 1} columns\n")

# ============================================================================
# File 2: Basic column descriptions
# ============================================================================

basic_descriptions = pd.DataFrame([
    {'column_name': 'translon_id', 'description': 'Unique identifier for the translon',
     'interpretation': 'Format: c{chrom}norep{n} or c{chrom}riboseqorf{n}',
     'key_for_story': 'Identifier for tracking specific candidates'},
    {'column_name': 'chrom', 'description': 'Chromosome name',
     'interpretation': 'chr1-chr22, chrX, chrY',
     'key_for_story': 'Genomic location'},
    {'column_name': 'start', 'description': 'Genomic start position (0-based)',
     'interpretation': 'BED format convention',
     'key_for_story': 'Genomic location'},
    {'column_name': 'end', 'description': 'Genomic end position (exclusive)',
     'interpretation': 'BED format convention',
     'key_for_story': 'Genomic location'},
    {'column_name': 'exonic_length', 'description': 'Total exonic sequence length in bp',
     'interpretation': 'Sum of all exon blocks',
     'key_for_story': 'Size of the translon feature'},
    {'column_name': 'blockCount', 'description': 'Number of exons in the translon',
     'interpretation': '1 = single exon, >1 = spliced multi-exon',
     'key_for_story': 'Structure complexity'},
    {'column_name': 'feature_mean', 'description': 'Mean PhyloP conservation score across translon',
     'interpretation': f'>1.5 = strong conservation, 0.5-1.5 = moderate, -0.5 to 0.5 = neutral, <-0.5 = depleted',
     'key_for_story': 'PRIMARY: Measures how conserved the translon is'},
    {'column_name': 'upstream_mean', 'description': 'Mean PhyloP score in 150bp upstream EXONIC flank',
     'interpretation': 'Conservation in exonic sequence before translon (walks along transcript exons, skips introns)',
     'key_for_story': 'Used to assess if conservation is regional or restricted'},
    {'column_name': 'downstream_mean', 'description': 'Mean PhyloP score in 150bp downstream EXONIC flank',
     'interpretation': 'Conservation in exonic sequence after translon (walks along transcript exons, skips introns)',
     'key_for_story': 'Used to assess if conservation is regional or restricted'},
    {'column_name': 'conservation_specificity', 'description': 'feature_mean - max(upstream_mean, downstream_mean)',
     'interpretation': '>0 = feature more conserved than both flanks; larger values = more specific conservation',
     'key_for_story': f'KEY METRIC: High specificity with low flanks (≤{FLANK_THRESHOLD}) = restricted conservation'},
    {'column_name': 'has_cds_overlap', 'description': 'Does translon overlap protein-coding CDS?',
     'interpretation': 'True = overlaps known gene (conservation likely explained), False = independent of CDS',
     'key_for_story': 'Filters out translons where conservation is from overlapping genes'},
    {'column_name': 'cds_overlap_proportion', 'description': 'Proportion of translon overlapping CDS',
     'interpretation': '0.0-1.0; proportion of translon bases that overlap protein-coding sequence',
     'key_for_story': 'Quantifies extent of CDS overlap'},
    {'column_name': 'overlapping_genes', 'description': 'Names of overlapping genes (if any)',
     'interpretation': 'Comma-separated gene names, or "-1" if no overlap',
     'key_for_story': 'Identifies which genes explain conservation (if has_cds_overlap=True)'},
    {'column_name': 'category', 'description': 'Classification for storytelling',
     'interpretation': f'"restricted_conservation" (top {len(restricted)}) or "other" (remaining {len(df_results) - len(restricted):,})',
     'key_for_story': f'STORY: The {len(restricted)} restricted_conservation translons are high-confidence novel candidates'}
])

output_basic_desc = OUTPUT_DIR / 'v3_story_column_descriptions.tsv'
basic_descriptions.to_csv(output_basic_desc, sep='\t', index=False)

print(f"2. BASIC COLUMN DESCRIPTIONS: {output_basic_desc}")
print(f"   - Describes {len(basic_descriptions)} columns in the story file\n")

# ============================================================================
# File 3: Already saved (comprehensive results)
# ============================================================================

print(f"3. COMPREHENSIVE RESULTS: {comprehensive_output}")
print(f"   - All {len(df_results):,} translons with full statistics\n")

# ============================================================================
# File 4: Comprehensive column descriptions
# ============================================================================

comprehensive_descriptions = pd.DataFrame([
    {'column_name': 'translon_id', 'description': 'Unique identifier for the translon',
     'value_type': 'string', 'notes': 'Format: c{chrom}norep{n} or c{chrom}riboseqorf{n}'},
    {'column_name': 'chrom', 'description': 'Chromosome name',
     'value_type': 'string', 'notes': 'chr1-chr22, chrX, chrY, chrM'},
    {'column_name': 'start', 'description': 'Genomic start position (0-based)',
     'value_type': 'integer', 'notes': 'BED format convention'},
    {'column_name': 'end', 'description': 'Genomic end position (exclusive)',
     'value_type': 'integer', 'notes': 'BED format convention'},
    {'column_name': 'strand', 'description': 'Genomic strand',
     'value_type': 'string', 'notes': '+ or -'},
    {'column_name': 'transcript_id', 'description': 'GENCODE transcript ID containing this translon',
     'value_type': 'string', 'notes': 'ENST format; "no_annotation" if not mapped'},
    {'column_name': 'exonic_length', 'description': 'Total length of all exonic sequence (bp)',
     'value_type': 'integer', 'notes': 'Sum of all block sizes'},
    {'column_name': 'blockCount', 'description': 'Number of exons/blocks in the translon',
     'value_type': 'integer', 'notes': '1 = single exon, >1 = spliced'},
    {'column_name': 'flank_size', 'description': 'Size of flanking regions analyzed (bp)',
     'value_type': 'integer', 'notes': f'{FLANK_SIZE}bp of exonic sequence extracted (transcript-aware, skips introns)'},
    {'column_name': 'flank_type', 'description': 'Type of flanking region extraction used',
     'value_type': 'string', 'notes': '"exonic" = walks along transcript exons, skips introns'},
    {'column_name': 'feature_n_bases', 'description': 'Number of bases with PhyloP scores in feature',
     'value_type': 'integer', 'notes': 'May be less than exonic_length if some positions lack scores'},
    {'column_name': 'feature_mean', 'description': 'Mean PhyloP score across translon',
     'value_type': 'float', 'notes': 'Positive = conserved, negative = depleted, ~0 = neutral'},
    {'column_name': 'feature_median', 'description': 'Median PhyloP score across translon',
     'value_type': 'float', 'notes': 'More robust to outliers than mean'},
    {'column_name': 'feature_std', 'description': 'Standard deviation of PhyloP scores',
     'value_type': 'float', 'notes': 'Measures variation in conservation across feature'},
    {'column_name': 'feature_min', 'description': 'Minimum PhyloP score in translon',
     'value_type': 'float', 'notes': 'Most depleted position'},
    {'column_name': 'feature_max', 'description': 'Maximum PhyloP score in translon',
     'value_type': 'float', 'notes': 'Most conserved position'},
    {'column_name': 'upstream_n_bases', 'description': 'Number of bases with PhyloP scores upstream',
     'value_type': 'integer', 'notes': 'From exonic sequence walking along transcript (skips introns)'},
    {'column_name': 'upstream_mean', 'description': 'Mean PhyloP score in upstream exonic flank',
     'value_type': 'float', 'notes': f'Conservation in {FLANK_SIZE}bp exonic sequence before translon'},
    {'column_name': 'upstream_median', 'description': 'Median PhyloP score in upstream exonic flank',
     'value_type': 'float', 'notes': ''},
    {'column_name': 'upstream_std', 'description': 'Standard deviation in upstream exonic flank',
     'value_type': 'float', 'notes': ''},
    {'column_name': 'upstream_min', 'description': 'Minimum PhyloP score in upstream exonic flank',
     'value_type': 'float', 'notes': ''},
    {'column_name': 'upstream_max', 'description': 'Maximum PhyloP score in upstream exonic flank',
     'value_type': 'float', 'notes': ''},
    {'column_name': 'downstream_n_bases', 'description': 'Number of bases with PhyloP scores downstream',
     'value_type': 'integer', 'notes': 'From exonic sequence walking along transcript (skips introns)'},
    {'column_name': 'downstream_mean', 'description': 'Mean PhyloP score in downstream exonic flank',
     'value_type': 'float', 'notes': f'Conservation in {FLANK_SIZE}bp exonic sequence after translon'},
    {'column_name': 'downstream_median', 'description': 'Median PhyloP score in downstream exonic flank',
     'value_type': 'float', 'notes': ''},
    {'column_name': 'downstream_std', 'description': 'Standard deviation in downstream exonic flank',
     'value_type': 'float', 'notes': ''},
    {'column_name': 'downstream_min', 'description': 'Minimum PhyloP score in downstream exonic flank',
     'value_type': 'float', 'notes': ''},
    {'column_name': 'downstream_max', 'description': 'Maximum PhyloP score in downstream exonic flank',
     'value_type': 'float', 'notes': ''},
    {'column_name': 'conservation_specificity', 'description': 'Difference: feature_mean - max(upstream_mean, downstream_mean)',
     'value_type': 'float', 'notes': 'Positive = feature more conserved than both flanks; key metric for restricted conservation'},
    {'column_name': 'has_cds_overlap', 'description': 'Boolean flag for CDS overlap',
     'value_type': 'boolean', 'notes': 'True if translon overlaps annotated protein-coding CDS'},
    {'column_name': 'same_strand', 'description': 'Is overlapping CDS on same strand?',
     'value_type': 'boolean', 'notes': 'Only meaningful if has_cds_overlap=True'},
    {'column_name': 'cds_overlap_proportion', 'description': 'Proportion of translon overlapping CDS',
     'value_type': 'float', 'notes': 'Range 0.0-1.0; fraction of translon bases overlapping protein-coding sequence'},
    {'column_name': 'overlapping_genes', 'description': 'Comma-separated list of overlapping gene names',
     'value_type': 'string', 'notes': '"-1" means no overlap (PyRanges sentinel value)'}
])

output_comprehensive_desc = OUTPUT_DIR / 'v3_comprehensive_column_descriptions.tsv'
comprehensive_descriptions.to_csv(output_comprehensive_desc, sep='\t', index=False)

print(f"4. COMPREHENSIVE COLUMN DESCRIPTIONS: {output_comprehensive_desc}")
print(f"   - Describes all {len(comprehensive_descriptions)} columns in comprehensive file\n")

print("="*80)
print("ALL OUTPUT FILES GENERATED")
print("="*80)

## Summary

In [ ]:
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print(f"\nTotal translons analyzed: {len(df_results):,}")
print(f"\nConservation breakdown:")
print(f"  Strongly conserved (PhyloP >{CONSERVATION_THRESHOLD}): {len(conserved):,} ({100*len(conserved)/len(df_results):.1f}%)")
print(f"    - With CDS overlap: {conserved['has_cds_overlap'].sum():,} ({100*conserved['has_cds_overlap'].sum()/len(conserved):.1f}%)")
print(f"    - Without CDS overlap: {len(conserved_no_cds):,} ({100*len(conserved_no_cds)/len(conserved):.1f}%)")
print(f"      • Regional (exonic flanks >{FLANK_THRESHOLD}): {len(regional):,} ({100*len(regional)/len(conserved_no_cds):.1f}%)")
print(f"      • RESTRICTED (exonic flanks ≤{FLANK_THRESHOLD}): {len(restricted):,} ({100*len(restricted)/len(conserved_no_cds):.1f}%)")

print(f"\n" + "="*80)
print("KEY FINDING FOR MANUSCRIPT")
print("="*80)
print(f"\n{len(restricted):,} translons ({100*len(restricted)/len(df_results):.1f}%) show conservation that is:")
print(f"  1. Strong (PhyloP >{CONSERVATION_THRESHOLD})")
print(f"  2. Independent of known protein-coding genes")
print(f"  3. Specific to the translon (exonic flanks ≤{FLANK_THRESHOLD})")

print(f"\nThese {len(restricted)} candidates are listed at the top of:")
print(f"  {output_basic}")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)